# Part 8 — Making Retrieval Smarter

*First-pass retrieval is fast but only roughly right. We sharpen it with three levers — before, during, and after retrieval — and the star is reranking.*

📖 Read the essay: https://www.mefby.com/essays/making-retrieval-smarter

🐍 Companion script: `rag_rerank.py`

**What you'll build:**

- A toy corpus where every chunk carries **metadata** (`section`, `year`).
- The **DURING lever** — metadata *pre-filtering* that shrinks the candidate set before scoring.
- Stage 1, the **wide net** — fast first-pass bi-encoder retrieval (cosine).
- Stage 2, the **rerank** — a cross-encoder that reads `(query, chunk)` together and reorders the candidates.
- The full **two-stage retrieve-then-rerank** pipeline, plus a tour of the **BEFORE lever** (query transformations).

> This notebook runs **fully offline** — numpy only. The real models (a SentenceTransformer bi-encoder and a CrossEncoder reranker) are used automatically if they're installed and cached; otherwise a transparent, deterministic stand-in keeps every cell runnable. You'll see a clear label whenever a fallback is active.

Previous: **Part 7 — Retrieval Deep Dive** · Next: **Part 9 — Advanced Retrieval Patterns**

## Setup

We need only `numpy`. We also pull in `hashlib` and `re` for the offline fallback embedder — the exact deterministic stand-in from Part 2, so this notebook never touches the network.

In [ ]:
import hashlib
import re

import numpy as np

np.set_printoptions(precision=3, suppress=True)
print('numpy', np.__version__, '— ready, no network needed')

## Where Part 7 left us, and three levers to pull

In **Part 7 — Retrieval Deep Dive** we made retrieval recall-strong with hybrid search, and ended on an uncomfortable truth: the ranked list is only *roughly* right. The chunk that best answers the question can sit at rank five or six, below something that merely *looks* relevant.

Every weakness here has a named fix, and the fixes fall into three positions around the retrieval step. I'll call them the three **levers**, and we pull them in pipeline order:

- **BEFORE — query transformations.** The user's raw question is often a poor search query. Rewrite it into something the index can match well.
- **DURING — metadata filtering.** Constrain the search to chunks that satisfy hard criteria (right section, right year, right tenant) *before* you ever score them.
- **AFTER — reranking.** Take the wide, roughly-ordered candidate set and reorder it with a slower, far more accurate model, then keep only the best few.

Improve the query going in, narrow the field, then sharpen the order coming out. Reranking is the star — so first let's understand *why* we need it.

## Why retrieval is only "roughly right": the bi-encoder bottleneck

A **bi-encoder** is the workhorse of ordinary dense retrieval: the query and each chunk are embedded *separately* into vectors, and relevance is the cosine similarity between them. The "bi" is the whole point — there are **two independent encoding passes that never see each other**. The chunk is embedded once, offline, when you build the index; the query is embedded on its own at search time; then you compare two finished vectors with cheap arithmetic.

That design is what makes retrieval *fast*: every chunk vector is precomputed, so a query is one embedding plus a nearest-neighbor lookup, even across millions of chunks.

The cost is precision, and it is **structural**, not a bug you can tune away. Each text is crushed into a single fixed-length vector *before* it has any idea what it will be compared against. The chunk vector can't emphasize the part that happens to matter for *your* question; the query vector can't lean on a specific phrase in a specific chunk. So a chunk that merely shares surface vocabulary (both mention "jacket") can outscore the chunk that actually answers the question. The ranking is plausible, not precise — and closing that gap is exactly what reranking does.

## The bi-encoder, guarded for offline use

The intended bi-encoder is the same one from Part 2 and Part 6:

```python
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')   # 384 dims, local
vectors = embedder.encode(texts, normalize_embeddings=True)  # unit vectors -> dot == cosine
```

That's the **headline path** — what you'd ship. But loading it needs a model download, so we wrap it in `try/except`. If the real model can't be loaded (no package, no network, no cached weights), we drop to a transparent, fully deterministic **hashing stand-in**: it mimics the *interface* (text in → fixed-length dense unit vector out) so the shape of the pipeline stays runnable and inspectable. The stand-in is honest — it's a hash of shared words, so it does **not** understand synonyms — but it keeps every cell alive offline.

In [ ]:
def load_real_embedder():
    """Return a real SentenceTransformer encode fn, or None if it can't load offline."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer('all-MiniLM-L6-v2')   # 384 dims, the INTENDED path

        def encode(texts):
            # normalize_embeddings=True -> unit vectors, so dot product == cosine.
            return np.asarray(model.encode(texts, normalize_embeddings=True))

        return encode
    except Exception as exc:   # missing package, no network, no cached weights...
        print(f'[real embedder unavailable: {type(exc).__name__}] -> offline fallback')
        return None

The fallback below is the exact deterministic embedder from Part 2: hash each token into the vector's dimensions and accumulate, then average and normalize to unit length. Same text → same vector, always. Texts that **share words** land closer, so it behaves a little like bag-of-words projected into a short dense vector — enough to demonstrate the *shape* and the geometry while staying fully offline.

In [ ]:
FALLBACK_DIM = 384   # mirror all-MiniLM-L6-v2's 384 dims on purpose

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def unit(vec):
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def _hashed_token_vector(token, dim):
    # Deterministic hash -> a reproducible pseudo-random direction for this token.
    h = hashlib.sha256(token.encode('utf-8')).digest()
    raw = np.frombuffer((h * (dim // len(h) + 1))[:dim], dtype=np.uint8)
    return (raw.astype(np.float64) / 255.0) * 2.0 - 1.0

def _fallback_encode_one(text, dim=FALLBACK_DIM):
    tokens = tokenize(text)
    if not tokens:
        return np.zeros(dim)
    vec = np.zeros(dim)
    for token in tokens:
        vec += _hashed_token_vector(token, dim)
    vec /= len(tokens)   # average so length doesn't depend on token count
    return unit(vec)     # unit length -> dot product == cosine

def fallback_encode(texts):
    return np.vstack([_fallback_encode_one(t) for t in texts])

Now pick the embedder: real model if it loads, transparent fallback otherwise. Everything downstream calls `embed(...)` and never has to care which one is active.

In [ ]:
_real_encode = load_real_embedder()
USING_REAL_EMBEDDER = _real_encode is not None
embed = _real_encode if USING_REAL_EMBEDDER else fallback_encode

label = 'REAL all-MiniLM-L6-v2' if USING_REAL_EMBEDDER else 'offline hashing fallback'
print(f'Bi-encoder in use: {label}')

## The corpus, now with metadata

This is the Part 6 corpus, but every chunk now carries **metadata** — a `section` and a `year`. Metadata is what the DURING lever filters on. Note the trap we've planted: the *winter jacket sale* chunk shares the word "jacket" with our query but is in the `promotions` section, while the chunk that genuinely answers it (*worn clothing counts as used*) is in `returns`.

In [ ]:
CORPUS = [
    {'text': 'Refunds are accepted within 30 days of purchase, provided the item is unused.', 'section': 'returns', 'year': 2024},
    {'text': 'Items marked final sale cannot be returned or exchanged.', 'section': 'returns', 'year': 2024},
    {'text': 'Standard shipping takes 3 to 5 business days; express is next-day.', 'section': 'shipping', 'year': 2024},
    {'text': 'To start a return, email support@example.com with your order number.', 'section': 'returns', 'year': 2023},
    {'text': 'Worn or washed clothing counts as used and is not eligible for a refund.', 'section': 'returns', 'year': 2024},
    {'text': 'Our winter jacket collection is on sale through the end of the month.', 'section': 'promotions', 'year': 2024},
    {'text': 'Gift cards never expire and are non-refundable.', 'section': 'giftcards', 'year': 2022},
    {'text': 'All electronics include a one-year limited warranty.', 'section': 'warranty', 'year': 2024},
    {'text': 'Shipping fees are non-refundable; late returns get store credit.', 'section': 'returns', 'year': 2024},
    {'text': 'Exchanges for a different size are free within the return window.', 'section': 'returns', 'year': 2024},
]

print(f'{len(CORPUS)} chunks, each with section + year metadata')
for c in CORPUS[:3]:
    print(' ', c['section'], '|', c['year'], '|', c['text'])

## Embed and store (Part 6, unchanged)

Each chunk keeps its metadata in `chunks`, and we precompute the parallel matrix of chunk vectors in `vectors`. This is the bi-encoder's gift: chunks are embedded **once**, offline, so query time is cheap. Because the vectors are unit length, a dot product reads straight off as cosine similarity.

In [ ]:
chunks = [dict(c) for c in CORPUS]                  # the 'store': text + metadata
vectors = embed([c['text'] for c in chunks])        # the parallel vector matrix

print('chunks stored:', len(chunks))
print('vectors shape:', np.asarray(vectors).shape, '(one row per chunk)')

## The DURING lever: metadata filtering

**Metadata filtering** constrains retrieval to only the chunks that match hard, non-negotiable criteria, so the similarity search runs over a smaller, cleaner pool. Only 2024 documents. Only this customer's files. Only the `returns` section. Why bother when similarity search already ranks things?

- **Precision** — a filter removes confidently-irrelevant matches outright instead of hoping a low score buries them.
- **Freshness** — embeddings have no sense of time; a `year` filter does.
- **Security and multi-tenancy** — *not optional*. An access-level filter is what stops one user's query from ever retrieving another user's documents. Part 12 is devoted to doing this properly.
- **A smaller, cheaper search space** — fewer candidates means faster, cheaper retrieval at scale.

We implement **pre-filtering**: apply the constraint *before* scoring, so we score only the chunks that already passed. `matching_indices` returns the surviving row indices so we can slice the vector matrix.

In [ ]:
def matching_indices(where):
    # `where` is a dict of metadata field -> required value, e.g. {'section': 'returns'}.
    return [i for i, c in enumerate(chunks)
            if all(c.get(field) == value for field, value in where.items())]

returns_only = matching_indices({'section': 'returns'})
print('returns-section chunk indices:', returns_only)
print('that filter keeps', len(returns_only), 'of', len(chunks), 'chunks')

A quick aside on **pre- vs post-filtering**. Pre-filtering (what we do) scores only the matching chunks, so you get the top-k *of the matching set*. Post-filtering runs the similarity search first and then drops the hits that fail the filter — which can quietly return *fewer than k* results, or none, even when plenty of matching chunks exist, because it filters a list that was already trimmed to the top overall matches. Prefer pre-filtering; most vector databases support it directly.

## Stage 1 — the wide net: fast first-pass retrieval

This is exactly Part 6's `retrieve`, with two tweaks: we fetch a **large `n`** (the wide net — say 10, where a real corpus would pull 50–100) instead of the final `k`, and we can optionally restrict it to chunks that pass the metadata filter. Its job is **recall**: get the right chunk *into the net*, order be damned.

In [ ]:
def first_pass(query, n=10, where=None):
    pool = matching_indices(where) if where else list(range(len(chunks)))
    q = embed([query])[0]
    scores = vectors[pool] @ q                  # cosine against the filtered pool
    order = np.argsort(-scores)[:n]             # best n within the pool
    return [{'text': chunks[pool[j]]['text'],
             'section': chunks[pool[j]]['section'],
             'score': float(scores[j])} for j in order]

QUERY = "Can I get a refund on a jacket I've already worn?"
wide = first_pass(QUERY, n=10)
print(f'QUERY: {QUERY}\n')
print('STAGE 1, first-pass order (bi-encoder cosine, top 10):')
for rank, c in enumerate(wide, 1):
    print(f'  {rank:>2}. {c["score"]:.2f}  {c["text"]}')

Look at the trap in action. The off-topic *winter jacket sale* ad scores high because it shares the word "jacket," while the chunk that actually settles the question — *worn clothing counts as used* — is stranded down the list, below the cut a naive top-three would take. The bi-encoder did its job (the right chunk is *in the net*), but the order is wrong. That's the looseness reranking fixes.

*(With the offline hashing fallback the exact numbers and order differ from the essay — the stand-in only matches shared words, it doesn't understand meaning. The point is the mechanism, not the digits.)*

## The AFTER lever: reranking with a cross-encoder

This is the chapter's star, and it directly attacks the bi-encoder bottleneck.

A **cross-encoder** takes the query and one candidate chunk *together*, as a single joined input, and outputs one number: how relevant this chunk is to this query. Because both texts go through the model at once, every word of the query can attend to every word of the chunk and vice-versa. It's not comparing two pre-baked summaries; it's *reading the pair* and judging the match directly. That's why a cross-encoder is dramatically more accurate at relevance — and why **reranking** (reordering candidates by its score) fixes exactly the looseness we diagnosed.

So why not rerank *everything* and skip the bi-encoder? Because a cross-encoder is slow in exactly the way the bi-encoder is fast: it can't precompute anything (the score depends on the query), so it must run once per `(query, chunk)` pair, at query time. Wonderful at judging a short list; hopeless as a first-pass search over a whole corpus. The resolution is **two-stage retrieval**: retrieve wide and fast, then rerank narrow and accurate.

## The cross-encoder, guarded for offline use

The intended reranker is a local cross-encoder from `sentence-transformers`:

```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # check current model names
rel = reranker.predict([(query, chunk_text), ...])              # one relevance score per pair
```

That's the **headline path** — and exactly what you'd ship. (Reranking models and recommended names move fast and I have a knowledge cutoff, so verify the current model before you ship.) Loading it needs a download, so we guard it with `try/except`. When it can't load, we fall back to a transparent **lexical-overlap** stand-in: it scores each pair by how much vocabulary the query and chunk share. That is a crude proxy — a hash-free, dependency-free way to *score a pair as a unit* — and it's enough to make the rerank visibly reorder the list while staying fully offline. We label loudly when the stand-in is active.

In [ ]:
def load_real_reranker():
    """Return a real CrossEncoder.predict fn, or None if it can't load offline."""
    try:
        from sentence_transformers import CrossEncoder

        model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # the INTENDED path

        def predict(pairs):
            return np.asarray(model.predict(pairs))

        return predict
    except Exception as exc:   # missing package, no network, no cached weights...
        print(f'[real reranker unavailable: {type(exc).__name__}] -> offline fallback')
        return None

The lexical-overlap stand-in below is deliberately simple and transparent. For a `(query, chunk)` pair it counts shared word *types* and normalizes by the query length — a small, deterministic relevance proxy. Unlike the bi-encoder's two separate vectors, this score looks at the **pair together**, which is the conceptual heart of a cross-encoder, even if the real model is vastly more capable.

In [ ]:
def _lexical_overlap_score(query, text):
    # A transparent, dependency-free stand-in for a real cross-encoder: score a
    # (query, chunk) PAIR as a unit by how much vocabulary they share. Deterministic.
    q = set(tokenize(query))
    d = set(tokenize(text))
    if not q:
        return 0.0
    shared = len(q & d)
    # Reward overlap, with a small bonus for covering more of the query's words.
    return shared + 0.25 * (shared / len(q))

def fallback_predict(pairs):
    return np.array([_lexical_overlap_score(q, t) for q, t in pairs], dtype=float)

In [ ]:
_real_predict = load_real_reranker()
USING_REAL_RERANKER = _real_predict is not None
predict_relevance = _real_predict if USING_REAL_RERANKER else fallback_predict

label = 'REAL cross-encoder/ms-marco-MiniLM-L-6-v2' if USING_REAL_RERANKER else 'offline lexical-overlap fallback'
print(f'Cross-encoder in use: {label}')

## Stage 2 — the rerank

Now the rerank itself: build one `(query, chunk)` pair per candidate, score them all with the cross-encoder, sort by that score descending, and keep the best `top_k`. We attach each chunk's `rerank` score so you can see the new order.

(The real `ms-marco` cross-encoder outputs an *unbounded relevance logit*, not a tidy 0–1 number, so live scores look different from the essay's illustrative values. Only the resulting **order** is the point.)

In [ ]:
def rerank(query, candidates, top_k=3):
    pairs = [(query, c['text']) for c in candidates]    # one pair per candidate
    rel = predict_relevance(pairs)                      # a relevance score per pair
    ranked = sorted(zip(candidates, rel), key=lambda cr: cr[1], reverse=True)
    return [{**c, 'rerank': float(s)} for c, s in ranked[:top_k]]

final = rerank(QUERY, wide, top_k=3)
print('STAGE 2, after cross-encoder rerank (top 3 kept):')
for rank, c in enumerate(final, 1):
    print(f'  {rank:>2}. {c["rerank"]:.2f}  {c["text"]}')

The cross-encoder, reading each chunk *together* with the query, lifts the decisive *worn clothing* chunk toward the top and sinks the off-topic sale ad. The generator now receives the chunks that genuinely settle the question instead of being misled by surface vocabulary. That is the whole payoff of the AFTER lever.

*(Under the offline fallbacks the exact ranking can differ — the lexical stand-in rewards shared words, not true relevance. With the real cross-encoder installed, the worn-clothing chunk rises to the top as in the essay.)*

## Putting the two stages together

`smart_retrieve` is the headline pattern of this part — **two-stage retrieve-then-rerank** — in two lines: filter + retrieve wide, then the accurate trim. You get the bi-encoder's speed over the whole corpus and the cross-encoder's precision over the short list, paying the expensive model on only a few dozen pairs.

In [ ]:
def smart_retrieve(query, n=10, top_k=3, where=None):
    candidates = first_pass(query, n=n, where=where)   # Part 6 retrieve, but WIDE
    return rerank(query, candidates, top_k=top_k)       # then the accurate TRIM

best = smart_retrieve(QUERY, n=10, top_k=3)
print('smart_retrieve -> top 3 chunks handed to the generator:')
for rank, c in enumerate(best, 1):
    print(f'  {rank:>2}. rerank={c["rerank"]:.2f}  [{c["section"]}]  {c["text"]}')

## The DURING lever on its own

Reranking is one fix; the metadata filter is the other half — and often the cheaper one. Because each chunk carries a `section`, we can restrict the first pass to the `returns` policy, and the *winter jacket sale* ad never even competes. No model needed: the filter removes the distractor before scoring ever happens.

In [ ]:
filtered = first_pass(QUERY, n=3, where={'section': 'returns'})
print("With a metadata filter (section == 'returns'), the sale ad never even competes:")
for rank, c in enumerate(filtered, 1):
    print(f'  {rank:>2}. {c["score"]:.2f}  {c["text"]}')

## The BEFORE lever: query transformations

We've now exercised DURING (filter) and AFTER (rerank). The third lever sits **before** retrieval. Users don't write search queries — they write questions that are short, ambiguous, conversational, or secretly several questions at once. A **query transformation** rewrites the raw question into one or more better search queries. Each technique buys quality with extra LLM calls (more latency and cost), so reach for them where a *class* of query keeps failing, not by default. The four worth knowing:

- **Multi-query (query expansion)** — ask an LLM for several rephrasings, retrieve for each, then union and dedupe the hits. Mainly lifts **recall**.
- **HyDE (Hypothetical Document Embeddings)** — ask an LLM to *write a hypothetical answer*, embed *that*, and search with it; an answer resembles the target documents more than a terse question does. Caveat: a confidently wrong hypothetical can steer retrieval off course, so treat it as a search probe, never as content you'd show the user.
- **Step-back prompting** — generate a broader, more abstract version of the question to gather foundational context. For "is a worn linen blazer covered," a useful step-back is "what is our overall returns policy."
- **Query decomposition** — split a complex multi-part question into sub-questions, retrieve for each, then combine. A first taste of the agentic patterns in Part 10.

Generating these rewrites needs an LLM, which we won't call here (offline). But the *downstream* mechanics — run each query, then union and dedupe the hits — are pure retrieval we already have. Below we **simulate** a multi-query expansion with a hand-written set of rephrasings (standing in for the LLM's output) and show the union-and-dedupe step on our offline pipeline.

In [ ]:
# An LLM would GENERATE these rephrasings from the raw question. Offline, we hand-write
# them to demonstrate the downstream multi-query mechanics: retrieve each, union, dedupe.
rephrasings = [
    'Can I get a refund on a jacket I have already worn?',
    'Is worn clothing eligible for a refund?',
    'refund policy for used or worn items',
]

def multi_query_retrieve(queries, n_each=3):
    seen, merged = set(), []
    for q in queries:
        for hit in first_pass(q, n=n_each):
            if hit['text'] not in seen:        # dedupe across rephrasings
                seen.add(hit['text'])
                merged.append(hit)
    return merged

union = multi_query_retrieve(rephrasings, n_each=3)
print(f'{len(rephrasings)} rephrasings -> {len(union)} unique chunks after union + dedupe:')
for c in union:
    print(f'  [{c["section"]}]  {c["text"]}')

Different phrasings surface different chunks, so the union widens the net — that's multi-query lifting recall. In a full pipeline you'd hand this wider candidate set to the cross-encoder rerank, combining the BEFORE and AFTER levers.

## The headline demo (the companion script's `__main__`)

Finally, the exact demo from `rag_rerank.py`, run on the offline path so you see real output: first-pass order, then the cross-encoder rerank, then the metadata filter on its own. This is the whole part in one block.

In [ ]:
query = "Can I get a refund on a jacket I've already worn?"
print(f'QUERY: {query}\n')

wide = first_pass(query, n=10)
print('STAGE 1, first-pass order (bi-encoder cosine, top 10):')
for rank, c in enumerate(wide, 1):
    print(f'  {rank:>2}. {c["score"]:.2f}  {c["text"]}')

final = rerank(query, wide, top_k=3)
print('\nSTAGE 2, after cross-encoder rerank (top 3 kept):')
for rank, c in enumerate(final, 1):
    print(f'  {rank:>2}. {c["rerank"]:.2f}  {c["text"]}')

filtered = first_pass(query, n=3, where={'section': 'returns'})
print("\nWith a metadata filter (section == 'returns'), the sale ad never even competes:")
for rank, c in enumerate(filtered, 1):
    print(f'  {rank:>2}. {c["score"]:.2f}  {c["text"]}')

print('\nembedder:', 'REAL' if USING_REAL_EMBEDDER else 'fallback',
      '| reranker:', 'REAL' if USING_REAL_RERANKER else 'fallback')

## What you learned

- First-pass retrieval is only roughly right because it uses a **bi-encoder**: query and chunks are embedded separately, so each text is crushed to one vector that never gets to consider the other. Fast, but structurally imprecise.
- You can sharpen retrieval with three **levers** in pipeline order: **BEFORE** (transform the query), **DURING** (filter by metadata), **AFTER** (rerank the results).
- **Metadata filtering** enforces hard criteria (date, section, and critically access level for security/multi-tenancy) and shrinks the search space. Prefer **pre-filtering** over post-filtering.
- A **cross-encoder** scores the query and a chunk *together* and judges relevance far more accurately than a bi-encoder, but is too slow to run over a whole corpus.
- **Two-stage retrieval (retrieve-then-rerank)** resolves the tension: retrieve a wide net fast, rerank it with the cross-encoder, keep the best few. Add levers only where your failure analysis demands it — measure before adding complexity (Part 11).

Every model load in this notebook is guarded: the real bi-encoder and cross-encoder are the headline path, with a transparent deterministic stand-in (hashing embedder + lexical-overlap rerank) keeping every cell runnable offline.

## Next

We've sharpened *how we rank* chunks. **Part 9 — Advanced Retrieval Patterns** rethinks *what we retrieve* — parent-document retrieval, sentence-window retrieval, self-querying, and contextual compression — because the best thing to retrieve and the best thing to *show the model* are not always the same chunk.